In [5]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

In [9]:
base_dir = "../../Coding_and_Data/Ecog"   # root folder

X = []
y = []

for ecog_folder in sorted(os.listdir(base_dir)):
    if 'Ecog' not in ecog_folder:
        continue
    ecog_path = os.path.join(base_dir, ecog_folder)

    if not os.path.isdir(ecog_path):
        continue

    # channel1, channel2, channel3
    for channel_folder in sorted(os.listdir(ecog_path)):
        channel_path = os.path.join(ecog_path, channel_folder)

        if not os.path.isdir(channel_path):
            continue

        # Read all CSV files in this channel
        for fname in sorted(os.listdir(channel_path)):
            if fname.endswith(".csv"):
                fpath = os.path.join(channel_path, fname)

                # Load CSV (assuming no header)
                data = pd.read_csv(fpath, header=None).values  # shape: (num_rows, 1601)

                # Split into X (first 1600) and y (last column)
                x_temp = data[:,:-1]

                X.append(x_temp)
                y.append(data[:, -1])

# Convert lists to arrays
X_ecog_all = np.vstack(X)   # shape: (total_rows, 1600)
y_ecog_all = np.hstack(y)   # shape: (total_rows,)

print("X shape:", X_ecog_all.shape)
print("y shape:", y_ecog_all.shape)

X shape: (28032, 1600)
y shape: (28032,)


In [10]:
#Label 0 = no seizure
#Label 1 = pre-ictal
#Label 2 = seizure
#Replace all 2 with 1
y_ecog_all[y_ecog_all == 2] = 1

In [11]:
# Train/test split
X_ecog_train, X_ecog_temp, y_ecog_train, y_ecog_temp = train_test_split(
    X_ecog_all, y_ecog_all, test_size=0.25, random_state=42, stratify = y_ecog_all, shuffle=True
)

X_ecog_valid, X_ecog_test, y_ecog_valid, y_ecog_test = train_test_split(
    X_ecog_temp, y_ecog_temp, test_size = 0.4, random_state = 42, stratify = y_ecog_temp, shuffle = True
)

train_mean = X_ecog_train.mean(axis=0)
train_std = X_ecog_train.std(axis=0) + 1e-8

#Fit normalization on the train dataset
X_ecog_train = (X_ecog_train - train_mean) / train_std
X_ecog_valid = (X_ecog_valid - train_mean) / train_std
X_ecog_test  = (X_ecog_test  - train_mean) / train_std

# Convert to PyTorch tensors
X_ecog_train_tensor = torch.tensor(X_ecog_train, dtype=torch.float32).unsqueeze(-1)  
y_ecog_train_tensor = torch.tensor(y_ecog_train, dtype=torch.float32).unsqueeze(-1)

X_ecog_valid_tensor = torch.tensor(X_ecog_valid, dtype=torch.float32).unsqueeze(-1)  
y_ecog_valid_tensor = torch.tensor(y_ecog_valid, dtype=torch.float32).unsqueeze(-1)

X_ecog_test_tensor = torch.tensor(X_ecog_test, dtype=torch.float32).unsqueeze(-1)
y_ecog_test_tensor = torch.tensor(y_ecog_test, dtype=torch.float32).unsqueeze(-1)

# Datasets and loaders
train_ecog_dataset = TensorDataset(X_ecog_train_tensor, y_ecog_train_tensor)
valid_ecog_dataset = TensorDataset(X_ecog_valid_tensor, y_ecog_valid_tensor)
test_ecog_dataset = TensorDataset(X_ecog_test_tensor, y_ecog_test_tensor)

train_ecog_loader = DataLoader(train_ecog_dataset, batch_size=64, shuffle=True)
valid_ecog_loader = DataLoader(valid_ecog_dataset, batch_size=64, shuffle=False)
test_ecog_loader = DataLoader(test_ecog_dataset, batch_size=64, shuffle=False)

print("Number of train batches:", len(train_ecog_loader))
print("Number of validation batches:", len(valid_ecog_loader))
print("Number of test batches:", len(test_ecog_loader))

Number of train batches: 329
Number of validation batches: 66
Number of test batches: 44


In [13]:
#Save tensors
torch.save({
    "X_train": X_ecog_train_tensor,
    "y_train": y_ecog_train_tensor,
    "X_valid": X_ecog_valid_tensor,
    "y_valid": y_ecog_valid_tensor,
    "X_test":  X_ecog_test_tensor,
    "y_test":  y_ecog_test_tensor
}, "ecog_tensors_channel_indep.pt")